# BoxBunny Runner

Essential commands for building, running, and testing the boxing robot system.

**Quick Start:** Run cells 1-4 in order, then launch what you need.

---
## 1. Build & Setup
Build the ROS 2 workspace and seed demo data.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash

echo "=== Building Workspace ==="
colcon build --symlink-install 2>&1
echo ""
source install/setup.bash
echo "=== Packages ==="
ros2 pkg list 2>/dev/null | grep boxbunny || echo "(build failed)"
echo ""
echo "=== Seeding Demo Data ==="
python3 tools/demo_data_seeder.py

---
## 2. Run Tests

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 -m pytest tests/ -v --tb=short 2>&1

---
## 3. System Check
Hardware, dependencies, and model status in one view.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/hardware_check.py

---
## 4. Launch System
Starts all ROS nodes + IMU simulator + GUI.
Press **STOP** (interrupt) to shut everything down.

In [ ]:
%%bash --no-raise-error
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash

cleanup() {
    echo ""
    echo "=== Stopping BoxBunny ==="
    # Kill the entire process group (launch + all children)
    kill -- -$LAUNCH_PID 2>/dev/null
    sleep 1
    # Force kill any survivors
    kill -9 -- -$LAUNCH_PID 2>/dev/null
    # Catch anything that escaped the process group
    pkill -9 -f 'imu_simulator.py' 2>/dev/null
    pkill -9 -f 'gui_main' 2>/dev/null
    pkill -9 -f 'ros2.launch' 2>/dev/null
    fuser -k 8080/tcp 2>/dev/null
    echo "All processes and windows stopped."
}
trap cleanup EXIT INT TERM

echo "Launching BoxBunny in dev mode..."
# Start in its own process group so we can kill the whole tree
setsid ros2 launch boxbunny_core boxbunny_dev.launch.py &
LAUNCH_PID=$!
sleep 5

echo ""
echo "=== Active ROS Nodes ==="
ros2 node list 2>/dev/null || echo "(nodes still starting...)"
echo ""
echo "=== RUNNING — Press STOP to shut down ==="

wait $LAUNCH_PID

---
## 6. IMU Simulator
Opens the IMU pad simulator GUI with clickable buttons.
Click pads to send navigation and punch events via ROS topics.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash

export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms

echo "Launching IMU Simulator..."
echo "Pad mapping: LEFT=prev, RIGHT=next, CENTRE=enter, HEAD=back"
echo "Close the window to stop."
echo ""

python3 tools/imu_simulator.py 2>&1

---
## 6. GUI Test
Launches the main BoxBunny touchscreen GUI for visual inspection.
Close the window to end the test.

In [ ]:
%%bash --no-raise-error
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash

export QT_QPA_PLATFORM=xcb
export QT_QPA_PLATFORM_PLUGIN_PATH=$(python3 -c "import PySide6; print(PySide6.__path__[0])")/Qt/plugins/platforms
unset QT_PLUGIN_PATH

echo "Launching BoxBunny GUI..."
echo "Close the window to end the test."
echo ""

python3 -c "
import sys, os
sys.path.insert(0, 'src/boxbunny_gui')
os.environ.pop('QT_PLUGIN_PATH', None)
from boxbunny_gui.app import BoxBunnyApp
app = BoxBunnyApp()
app.run()
" 2>&1 || true

---
## 6. Phone Dashboard
Starts the dashboard server with a public tunnel URL and QR code.
Scan the QR with your phone to open the dashboard from anywhere.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/dashboard_tunnel.py

---
## 7. CV Model Live Test
Opens a camera window showing pose skeleton + predicted action labels.
Stand 1.5-2m from the camera and throw punches. Press **q** to stop.

In [ ]:
%%bash
set +e
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(D435i camera not connected or models missing)" 

---
## 8. LLM Coach Test
Loads the local Qwen 2.5-3B model and generates a sample coaching tip.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/test_llm.py

---
## 9. Build Vue Frontend
Rebuild the phone dashboard SPA after making frontend changes.

In [ ]:
%%bash
set +e
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
cd /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/src/boxbunny_dashboard/frontend
npm run build 2>&1

---
## 10. Sound Test
Play all sound effects to verify audio.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/play_sounds.py

---
## 11. Demo User Profiles
Visual cards for each demo user with XP, streaks, and records.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/view_user_dashboards.py

---
## 12. Benchmark Test
Percentile rankings for demo users against population norms.

In [ ]:
%run /home/boxbunny/Desktop/doomsday_integration/boxing_robot_ws/notebooks/scripts/benchmark_test.py